DenseNet-SWIN Hybrid Model

In [ ]:
#!/usr/bin/env python
import os
import sys
import time
import copy
import random
import pickle
import logging
import argparse
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
import timm
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score, classification_report)
from colorama import Fore, Style
import matplotlib.pyplot as plt

# ==============================================
#               Setup & Logging
# ==============================================
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logging.info(f"Using device: {device}")
if torch.cuda.is_available():
    logging.info(f"GPU: {torch.cuda.get_device_name(0)} - Allocated Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

green_bar_format = f"{Fore.GREEN}{{l_bar}}{{bar}}{{r_bar}}{Style.RESET_ALL}"

# ==============================================
#       Data Augmentation & Loading (Albumentations-based)
# ==============================================
import random
from albumentations import (
    HorizontalFlip, VerticalFlip, ShiftScaleRotate, CLAHE,
    RandomRotate90, Transpose, HueSaturationValue, GaussNoise,
    Sharpen, Emboss, RandomBrightnessContrast, OneOf, Compose
)
import numpy as np
from PIL import Image

random.seed(42)

def strong_aug(p=0.5):
    return Compose([
        RandomRotate90(p=0.2),
        Transpose(p=0.2),
        HorizontalFlip(p=0.5),
        VerticalFlip(p=0.5),
        OneOf([GaussNoise()], p=0.2),
        ShiftScaleRotate(p=0.2),
        OneOf([
            CLAHE(clip_limit=2),
            Sharpen(),
            Emboss(),
            RandomBrightnessContrast(),
        ], p=0.2),
        HueSaturationValue(p=0.2),
    ], p=p)

def augment(aug, image):
    return aug(image=image)["image"]

class Aug(object):
    def __call__(self, img):
        aug_inst = strong_aug(p=0.9)
        return Image.fromarray(augment(aug_inst, np.array(img)))

def normalize_data():
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    return {
        "train": transforms.Compose([
            Aug(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
        "valid": transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
        "test": transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])
    }

# ==============================================
#         Dataset Paths & Creation
# ==============================================
base_dataset_dir = "/kaggle/input/deepfake-dataset/deepfake_dataset"  

train_datasets = []
val_datasets = []
test_datasets = []

sub_datasets = ["celebdf", "dfdc", "ffpp", "realfake"]
data_transforms = normalize_data()

for sub in sub_datasets:
    # Train dataset
    train_path = os.path.join(base_dataset_dir, "train", sub)
    train_dataset = datasets.ImageFolder(train_path, transform=data_transforms["train"])
    train_datasets.append(train_dataset)

    # Validation dataset
    val_path = os.path.join(base_dataset_dir, "validation", sub)
    val_dataset = datasets.ImageFolder(val_path, transform=data_transforms["valid"])
    val_datasets.append(val_dataset)

    # Test dataset
    test_path = os.path.join(base_dataset_dir, "test", sub)
    test_dataset = datasets.ImageFolder(test_path, transform=data_transforms["test"])
    test_datasets.append(test_dataset)

full_train_dataset = ConcatDataset(train_datasets)
full_val_dataset = ConcatDataset(val_datasets)
test_dataset = ConcatDataset(test_datasets)

train_subset_size = 20000
train_indices = random.sample(range(len(full_train_dataset)), train_subset_size)
train_dataset = Subset(full_train_dataset, train_indices)
print(f"Using {len(train_dataset)} training samples.")

val_subset_size = 5000
val_indices = random.sample(range(len(full_val_dataset)), val_subset_size)
val_dataset = Subset(full_val_dataset, val_indices)
print(f"Using {len(val_dataset)} validation samples.")

def count_labels_from_subset(subset):
    all_targets = []
    for idx in subset.indices:
        for dataset in subset.dataset.datasets:
            if idx < len(dataset):
                label = dataset.targets[idx]
                all_targets.append(label)
                break
            else:
                idx -= len(dataset)
    return Counter(all_targets)

def count_labels_from_concat(concat_dataset):
    all_targets = []
    for dataset in concat_dataset.datasets:
        all_targets.extend(dataset.targets)
    return Counter(all_targets)

train_label_counts = count_labels_from_subset(train_dataset)
print(f"Train Dataset Class Counts: {train_label_counts}")

val_label_counts = count_labels_from_subset(val_dataset)
print(f"Validation Dataset Class Counts: {val_label_counts}")

test_label_counts = count_labels_from_concat(test_dataset)
print(f"Test Dataset Class Counts: {test_label_counts}")

batch_size = 16
num_workers = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True, persistent_workers=True)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True, persistent_workers=True)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

dataset_sizes = {
    "train": len(train_dataset),
    "validation": len(val_dataset),
    "test": len(test_dataset)
}

print(f"Dataset sizes -> Train: {dataset_sizes['train']}, Validation: {dataset_sizes['validation']}, Test: {dataset_sizes['test']}")

# ==============================================
#      Model Components & Enhanced Modules
# ==============================================
# Enhanced Attention Fusion with added non-linearity after projection.
class AttentionFusion(nn.Module):
    """
    Projects each branch's feature to a common dimension, applies a ReLU,
    then learns attention scores to fuse them.
    """
    def __init__(self, input_dims, common_dim=256):
        super(AttentionFusion, self).__init__()
        self.proj_layers = nn.ModuleList([nn.Linear(dim, common_dim) for dim in input_dims])
        self.relu = nn.ReLU()
        self.attn_fc = nn.Linear(common_dim, 1)  

    def forward(self, features):
        proj_features = [self.relu(proj(f)) for f, proj in zip(features, self.proj_layers)]
        scores = [self.attn_fc(f) for f in proj_features]
        scores = torch.cat(scores, dim=1)  
        weights = torch.softmax(scores, dim=1)
        fused = sum(weights[:, i:i+1] * proj_features[i] for i in range(len(proj_features)))
        return fused, weights

class FrequencyBranch(nn.Module):
    def __init__(self, output_dim=256):
        super(FrequencyBranch, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, output_dim)

    def forward(self, x):
        fft = torch.fft.fft2(x, norm='ortho')
        fft = torch.abs(fft)
        fft = torch.log(fft + 1.0)
        out = self.conv(fft)
        out = self.avgpool(out).view(out.size(0), -1)
        out = self.fc(out)
        return out

# Hybrid Model combining DenseNet161, Swin, and Frequency Branches.
class HybridDeepfakeDetector(nn.Module):
    def __init__(self, num_classes=1):
        super(HybridDeepfakeDetector, self).__init__()
        self.densenet = timm.create_model('densenet161', pretrained=True)
        if hasattr(self.densenet, 'classifier'):
            self.densenet.classifier = nn.Identity()
        
        self.swin = timm.create_model('swin_small_patch4_window7_224', pretrained=True)
        if hasattr(self.swin, 'head'):
            self.swin.head = nn.Identity()
        elif hasattr(self.swin, 'fc'):
            self.swin.fc = nn.Identity()

        dummy = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            densenet_feat = self.densenet(dummy)
            if densenet_feat.ndim == 4:
                densenet_feat = nn.functional.adaptive_avg_pool2d(densenet_feat, 1).view(1, -1)
            self.densenet_dim = densenet_feat.shape[1]
            
            swin_feat = self.swin.forward_features(dummy) if hasattr(self.swin, 'forward_features') else self.swin(dummy)
            if swin_feat.ndim == 4:
                swin_feat = nn.functional.adaptive_avg_pool2d(swin_feat, 1).view(1, -1)
            self.swin_dim = swin_feat.shape[1]
            
            logging.info(f"DenseNet161 feature dim: {self.densenet_dim}")
            logging.info(f"Swin feature dim: {self.swin_dim}")
            print(f"DenseNet161 feature dim: {self.densenet_dim}")
            print(f"Swin feature dim: {self.swin_dim}")

        self.freq_branch = FrequencyBranch(output_dim=256)
        freq_dim = 256

        fusion_input_dims = [self.densenet_dim, self.swin_dim, freq_dim]
        self.attn_fusion = AttentionFusion(input_dims=fusion_input_dims, common_dim=256)

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        densenet_feat = self.densenet(x)
        if densenet_feat.ndim == 4:
            densenet_feat = nn.functional.adaptive_avg_pool2d(densenet_feat, 1).view(x.size(0), -1)
        swin_feat = self.swin.forward_features(x) if hasattr(self.swin, 'forward_features') else self.swin(x)
        if swin_feat.ndim == 4:
            swin_feat = nn.functional.adaptive_avg_pool2d(swin_feat, 1).view(x.size(0), -1)
        freq_feat = self.freq_branch(x)
        fused_feat, attn_weights = self.attn_fusion([densenet_feat, swin_feat, freq_feat])
        out = self.classifier(fused_feat)
        return out, attn_weights

model = HybridDeepfakeDetector(num_classes=1)
if torch.cuda.device_count() > 1:
    logging.info(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
model = model.to(device)
print("Hybrid Deepfake Detector created.")

# ==============================================
#         Loss, Optimizer, Scheduler
# ==============================================
pos_weight_value = 10667 / 9333
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value]).to(device))
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=0)
num_epochs = 15  

steps_per_epoch = len(train_loader)
total_steps = num_epochs * steps_per_epoch
scheduler = OneCycleLR(optimizer, max_lr=1e-4, total_steps=total_steps, pct_start=0.1, anneal_strategy='cos')

# ==============================================
#         Threshold Optimization Function
# ==============================================
def optimize_threshold(probs, true_labels):
    best_thresh = 0.5
    best_acc = 0.0
    for t in np.arange(0.0, 1.0, 0.01):
        preds = (probs >= t).astype(int)
        acc = accuracy_score(true_labels, preds)
        if acc > best_acc:
            best_acc = acc
            best_thresh = t
    return best_thresh, best_acc

# ==============================================
#         Training & Validation Loops
# ==============================================
scaler = torch.amp.GradScaler('cuda')
best_val_loss = float('inf')
patience = 5
trigger_times = 0

train_history = {"loss": [], "acc": []}
val_history = {"loss": [], "acc": []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training", leave=False, bar_format=green_bar_format)
    for images, labels in train_bar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs, _ = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).int()
        train_correct += (preds.cpu() == labels.cpu().int()).sum().item()
        train_total += labels.size(0)
        current_loss = running_loss / train_total
        current_acc = train_correct / train_total
        train_bar.set_postfix(loss=f"{current_loss:.4f}", acc=f"{current_acc:.4f}")
    epoch_loss = running_loss / dataset_sizes["train"]
    train_acc = train_correct / train_total
    train_history["loss"].append(epoch_loss)
    train_history["acc"].append(train_acc)

    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_probs = []
    val_true = []
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation", leave=False, bar_format=green_bar_format)
    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            with torch.amp.autocast('cuda'):
                outputs, _ = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(outputs)
            val_probs.extend(probs.cpu().numpy().flatten())
            preds = (probs > 0.5).int()
            val_correct += (preds.cpu() == labels.cpu().int()).sum().item()
            val_total += labels.size(0)
            val_true.extend(labels.cpu().numpy().flatten())
            val_bar.set_postfix(loss=f"{loss.item():.4f}")
    epoch_val_loss = val_loss / dataset_sizes["validation"]
    epoch_val_acc = val_correct / val_total
    val_history["loss"].append(epoch_val_loss)
    val_history["acc"].append(epoch_val_acc)
    
    best_thresh, best_val_acc = optimize_threshold(np.array(val_probs), np.array(val_true))
    
    val_preds = (np.array(val_probs) >= best_thresh).astype(int)
    val_accuracy = accuracy_score(val_true, val_preds)
    val_precision = precision_score(val_true, val_preds)
    val_recall = recall_score(val_true, val_preds)
    val_f1 = f1_score(val_true, val_preds)
    try:
        val_auc = roc_auc_score(val_true, val_probs)
    except Exception:
        val_auc = 0.0

    logging.info(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {epoch_loss:.4f}, Train Acc: {train_acc:.4f}")
    logging.info(f"                Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}, Best Thresh: {best_thresh:.2f}")
    logging.info(f"                Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {epoch_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"                Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}, Best Thresh: {best_thresh:.2f}")
    print(f"                Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        trigger_times = 0
        torch.save(model.state_dict(), "best_hybrid_model.pth")
    else:
        trigger_times += 1
        if trigger_times >= patience:
            logging.info("Early stopping triggered!")
            break

# ==============================================
#         Plot Training History
# ==============================================
def save_plot(metric_history, metric_name, mod):
    plt.figure(figsize=(10,5))
    plt.plot(metric_history, label=f"{metric_name}")
    plt.title(f"{metric_name} Trend")
    plt.xlabel("Epoch")
    plt.ylabel(metric_name)
    plt.legend()
    os.makedirs("result/figs", exist_ok=True)
    filename = os.path.join("result/figs", f"{mod}_{metric_name}_trend_{time.strftime('%b_%d_%Y_%H_%M_%S')}.png")
    plt.savefig(filename)
    plt.close()
    print(f"{metric_name} plot saved as {filename}")

save_plot(train_history["loss"], "Train Loss", "hybrid")
save_plot(train_history["acc"], "Train Accuracy", "hybrid")
save_plot(val_history["loss"], "Validation Loss", "hybrid")
save_plot(val_history["acc"], "Validation Accuracy", "hybrid")

# ==============================================
#         Testing & Evaluation
# ==============================================
model.load_state_dict(torch.load("best_hybrid_model.pth"))
model.eval()

test_probs = []
all_preds = []
all_labels = []
test_loss = 0.0
test_correct = 0
test_total = 0
test_bar = tqdm(test_loader, desc="Testing", leave=False, bar_format=green_bar_format)
with torch.no_grad():
    for images, labels in test_bar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        with torch.amp.autocast('cuda'):
            outputs, _ = model(images)
            loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(outputs)
        test_probs.extend(probs.cpu().numpy().flatten())
        preds = (probs > best_thresh).int()
        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())
        test_correct += (preds.cpu() == labels.cpu().int()).sum().item()
        test_total += labels.size(0)
        test_bar.set_postfix(loss=f"{loss.item():.4f}")
test_loss /= dataset_sizes["test"]
test_acc = test_correct / test_total

final_preds = (np.array(test_probs) >= best_thresh).astype(int)
final_labels = np.array(all_labels).astype(int)

accuracy_metric = accuracy_score(final_labels, final_preds)
precision_metric = precision_score(final_labels, final_preds)
recall_metric = recall_score(final_labels, final_preds)
f1_metric = f1_score(final_labels, final_preds)
conf_matrix = confusion_matrix(final_labels, final_preds)
try:
    auc_metric = roc_auc_score(final_labels, test_probs)
except Exception:
    auc_metric = "N/A"
classification_rep = classification_report(final_labels, final_preds)

print("\n=== Test Evaluation Metrics ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {accuracy_metric:.4f}")
print(f"Precision: {precision_metric:.4f}")
print(f"Recall: {recall_metric:.4f}")
print(f"F1 Score: {f1_metric:.4f}")
print(f"ROC-AUC: {auc_metric}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(classification_rep)


Using 20000 training samples.
Using 5000 validation samples.
Train Dataset Class Counts: Counter({0: 10633, 1: 9367})
Validation Dataset Class Counts: Counter({0: 2675, 1: 2325})
Test Dataset Class Counts: Counter({0: 4470, 1: 3874})
Dataset sizes -> Train: 20000, Validation: 5000, Test: 8344


model.safetensors:   0%|          | 0.00/116M [00:00<?, ?B/s]

DenseNet161 feature dim: 2208
Swin feature dim: 7
Hybrid Deepfake Detector created.


Epoch 1/15 - Training:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 1/15 - Validation:   0%|          | 0/313 [00:00<?, ?it/s]

...

Epoch 15/15 - Validation:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch [15/15] - Train Loss: 0.2013, Train Acc: 0.9325
                Val Loss: 0.2098, Val Acc: 0.9222, Best Thresh: 0.53
                Precision: 0.9172, Recall: 0.9196, F1: 0.9184, AUC: 0.9775
Train Loss plot saved as result/figs/hybrid_Train Loss_trend_Mar_31_2025_15_34_13.png
Train Accuracy plot saved as result/figs/hybrid_Train Accuracy_trend_Mar_31_2025_15_34_13.png
Validation Loss plot saved as result/figs/hybrid_Validation Loss_trend_Mar_31_2025_15_34_14.png
Validation Accuracy plot saved as result/figs/hybrid_Validation Accuracy_trend_Mar_31_2025_15_34_14.png


<ipython-input-2-ffbf2b5c094f>:470: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_hybrid_model.pth"))


Testing:   0%|          | 0/522 [00:00<?, ?it/s]


=== Test Evaluation Metrics ===
Test Loss: 0.2245
Test Accuracy: 0.9136
Precision: 0.9021
Recall: 0.9130
F1 Score: 0.9075
ROC-AUC: 0.9736756198323244
Confusion Matrix:
[[4086  384]
 [ 337 3537]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.91      0.92      4470
           1       0.90      0.91      0.91      3874

    accuracy                           0.91      8344
   macro avg       0.91      0.91      0.91      8344
weighted avg       0.91      0.91      0.91      8344

